In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.schema_validation.emi_data')

In [0]:
df.display()

In [0]:
df = df.withColumn("due_date", unix_timestamp(col("due_date")))

In [0]:
df.display()

In [0]:
df = df.withColumn("paid_at", unix_timestamp(col("paid_at")))

In [0]:
df.display()

In [0]:
df = df.withColumn(
    "mandate_type",
    regexp_extract(
        col("emi_details_json"),
        r'mandate_type"":""([^"]+)',
        1
    )
)

display(df.select("emi_details_json", "mandate_type"))

In [0]:
df.display()

In [0]:
from pyspark.sql import functions as F

df = (
    df
    .withColumn("created_at", F.col("created_at").cast("long") * 1000)
    .withColumn("updated_at", F.col("updated_at").cast("long") * 1000)
)

df.select("loan_id", "created_at", "updated_at").show(10, truncate=False)

In [0]:
df = df.withColumn("_ingested_at", unix_timestamp(col("_ingested_at").cast("timestamp")))
df.limit(5).show()

In [0]:
(df.write
 .format('delta')
 .mode('append')
 .saveAsTable('fintech360.silver.emi_data'))